In [2]:
import nest_asyncio
nest_asyncio.apply()

import numpy as np
import plotly.graph_objects as go
from ipywidgets import FloatSlider, HBox, VBox, ToggleButton
import asyncio
from IPython.display import display, HTML

C_dl = 3.0  # Double layer capacitance 
k_ads_co = 90.0  # Adsorption rate constant for CO 
k_oh_form = 8.0  # Rate constant for OH formation 
E_ref_oh = 0.6  # Reference potential for OH formation (V)
k_clean = 20.0  # Rate constant for surface cleaning 

class HORSimulation:
    def __init__(self):
        self.reset()
        
    def reset(self):
        self.t_data = [0.0] # Time data for plotting
        self.p_data = [0.0] # Potential data for plotting
        self.theta_oh_data = [0.0] # Surface coverage of OH for plotting
        self.theta_free_data = [1.0] # Surface coverage of free sites for plotting

        self.t = 0.0 # Simulation time
        self.potential = 0.0 # Initial potential (V)
        self.theta_oh = 0.0 # Initial surface coverage of OH 
        self.theta_co = 0.0 # Initial surface coverage of CO
        self.theta_free = 1.0 # Initial surface coverage of free sites

        self.dt = 0.003

    def step(self, current_val, co_val):
        j_target = current_val / 100.0 # Current scaling
        co_press = co_val / 100.0 # CO pressure scaling
        theta_free = max(1e-6, 1 - self.theta_co - self.theta_oh) 

        d_theta_co = k_ads_co * co_press * theta_free # Rate of CO adsorption
        rate_oh_formation = k_oh_form * np.exp((self.potential - E_ref_oh) / 0.033) * theta_free # Rate of OH formation
        r_clean = k_clean * self.theta_co * self.theta_oh # Rate of surface cleaning due to CO and OH interaction

        self.theta_co = np.clip(self.theta_co + (d_theta_co - r_clean) * self.dt, 0, 1) # Update CO surface coverage
        self.theta_oh = np.clip(self.theta_oh + (rate_oh_formation - r_clean) * self.dt, 0, 1) # Update OH surface coverage
        self.theta_free = max(1e-6, 1 - self.theta_co - self.theta_oh) # Update free site surface coverage

        j_hor = 0.1 * self.theta_free * np.sinh(self.potential / 0.07) # HOR current density based on free sites and potential
        dE_dt = (j_target - j_hor) / C_dl # Potential change rate based on current difference and double layer capacitance
        self.potential = np.clip(self.potential + dE_dt * self.dt, 0, 1) # Update potential with clipping to [0, 1] V

        self.t += self.dt # Increment simulation time
        # Store data for plotting
        self.t_data.append(self.t)
        self.p_data.append(self.potential)
        self.theta_oh_data.append(self.theta_oh)
        self.theta_free_data.append(self.theta_free)

        # Limit data length to prevent memory issues on mobile devices
        if len(self.t_data) > 6650:
            self.t_data.pop(0)
            self.p_data.pop(0)
            self.theta_oh_data.pop(0)
            self.theta_free_data.pop(0)

sim = HORSimulation()
fig = go.FigureWidget()

# Initial plot for potential and surface coverages
fig.add_scatter(
    x=sim.t_data,
    y=sim.p_data,
    mode='lines',
    line=dict(color='black', width=2),
    name='Potential',
    yaxis='y1'
)

fig.add_scatter(
    x=sim.t_data,
    y=[value * 100 for value in sim.theta_oh_data],
    mode='lines',
    line=dict(color='red', width=2),
    name='Θ (OH)',
    yaxis='y2'
)

fig.add_scatter(
    x=sim.t_data,
    y=[value * 100 for value in sim.theta_free_data],
    mode='lines',
    line=dict(color='blue', width=2),
    name='Θ (Free)',
    yaxis='y2'
)

# Adapt Layout for better mobile display
fig.update_layout(
    xaxis=dict(title=dict(text='Time', font=dict(size=18, color='black')), showticklabels=False, linecolor='black', linewidth=2, mirror=True, showline=True, showgrid=False, range=[0, 20]),
    yaxis=dict(
        title=dict(text='Potential (V)', font=dict(size=18, color='black')),
        range=[0, 1.0],
        showgrid=False,
        linecolor='black',
        linewidth=2,
        mirror=True,
        showline=True,
        ticks='outside',
        tickwidth=2,
        tickfont=dict(size=16, color='black')
    ),
    yaxis2=dict(
        title=dict(text='Surface coverage (%)', font=dict(size=18, color='black')),
        autorange=True,
        fixedrange=False,
        overlaying='y',
        side='right',
        showgrid=False,
        ticks='outside',
        tickwidth=2,
        tickfont=dict(size=16, color='black')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(x=0.01, y=0.99, font=dict(size=16, color='black')),
    margin=dict(l=50, r=50, t=50, b=50)
)

slider_layout = dict(width='350px', height='50px')
slider_style = {'description_width': 'initial'}

slider_co = FloatSlider(
    value=60, min=0, max=100, step=1,
    description='CO-Concentration:',
    layout=slider_layout,
    style=slider_style,
    readout_format='d'
)

slider_curr = FloatSlider(
    value=50, min=0, max=100, step=1,
    description='Current:',
    layout=slider_layout,
    style=slider_style,
    readout_format='d'
)

play_btn = ToggleButton(
    value=False,
    description='▶ Start',
    button_style='success',
    layout=dict(width='700px', height='60px')
)
# Custom CSS for better mobile display
display(HTML('''
    <style>
        .widget-label {
            font-size: 16px !important;
        }
        .widget-readout {
            font-size: 18px !important;
            color: #000000 !important;
        }
        .jupyter-button {
            font-size: 20px !important;
            font-weight: bold !important;
        }
        .slider-container {
            padding-top: 10px;
        }
    </style>
'''))

# Update loop for the simulation and plot
async def update_loop():
    while play_btn.value:
        # 1. Calculate multiple steps to reduce the number of updates
        for _ in range(60):
            sim.step(slider_curr.value, slider_co.value)

        # 2. Downsampling for display
        downsample_factor = 15
        show_t = sim.t_data[::downsample_factor]
        show_p = sim.p_data[::downsample_factor]
        show_oh = [v * 100 for v in sim.theta_oh_data[::downsample_factor]]
        show_free = [v * 100 for v in sim.theta_free_data[::downsample_factor]]

        last_t = sim.t_data[-1]

        # 3. Update plot
        with fig.batch_update():
            fig.data[0].x = show_t
            fig.data[0].y = show_p
            fig.data[1].x = show_t
            fig.data[1].y = show_oh
            fig.data[2].x = show_t
            fig.data[2].y = show_free            
            fig.layout.xaxis.range = [max(0, last_t - 20), max(20, last_t)]


        # wait before the next update 
        await asyncio.sleep(0.055)

# Play button event handler
def on_play_change(change):
    if change['new']:
        play_btn.description = '⏸ Stop'
        play_btn.button_style = 'danger'
        asyncio.create_task(update_loop())
    else:
        play_btn.description = '▶ Start'
        play_btn.button_style = 'success'

# Connect the play button to the event handler
play_btn.observe(on_play_change, names='value')
display(VBox([HBox([slider_co, slider_curr]), play_btn, fig]))